In [1]:
# Installing packages

!pip install transformers
!pip install accelerate

In [2]:
# Importing Libraries
import torch
import torch.nn as nn

from transformers import (
    AutoModel,
    AutoTokenizer
)

In [3]:
# Checking whether GPU is cuda
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [4]:
# Build Baseline Model
class XLMRClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            "xlm-roberta-base"
        )

        self.dropout = nn.Dropout(
            0.3
        )

        self.fc = nn.Linear(
            768,
            2
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

      #  cls_embedding = outputs.last_hidden_state[:,0,:]
        cls_embedding = outputs.last_hidden_state.mean(dim=1)

        x = self.dropout(
            cls_embedding
        )

        logits = self.fc(x)

        return logits

In [5]:
# Creating the model
model = XLMRClassifier().to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
# Loss function
criterion = nn.CrossEntropyLoss()

In [7]:
# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

Solving NameError: 'train_loader' was declared in Notebook 4, trying to use it in noebook 5 is reporting an error. This is how to solve it ...

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# Load dataset

import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/MultilingualFakeNews/data/english_clean_v2.csv"
)

df = df[["text", "target"]]

In [10]:
# Train / Validation Split
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["target"]
)

In [11]:
# Load tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "xlm-roberta-base"
)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [12]:
# Dataset class
import torch
from torch.utils.data import Dataset

class FakeNewsDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_len=384
    ):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids":
                encoding["input_ids"].flatten(),

            "attention_mask":
                encoding["attention_mask"].flatten(),

            "label":
                torch.tensor(
                    self.labels[idx],
                    dtype=torch.long
                )
        }

In [13]:
# Create datasets
train_dataset = FakeNewsDataset(
    train_df["text"].tolist(),
    train_df["target"].tolist(),
    tokenizer
)

val_dataset = FakeNewsDataset(
    val_df["text"].tolist(),
    val_df["target"].tolist(),
    tokenizer
)

In [14]:
# Create dataloaders
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8
)

In [15]:
# Veryfy...
print(len(train_loader))
print(len(val_loader))

59
15


In [16]:
# Test forward pass
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)

outputs = model(
    input_ids,
    attention_mask
)

print(outputs.shape)

torch.Size([8, 2])


In [17]:
# TRAINIG FUNCTION
def train_epoch(
    model,
    data_loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch in data_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids,
            attention_mask
        )

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(
            outputs,
            dim=1
        )

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

    accuracy = correct / total

    return (
        total_loss / len(data_loader),
        accuracy
    )

In [18]:
# VALIDATION FUNCTION
def eval_epoch(
    model,
    data_loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for batch in data_loader:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch[
                "attention_mask"
            ].to(device)

            labels = batch["label"].to(device)

            outputs = model(
                input_ids,
                attention_mask
            )

            loss = criterion(
                outputs,
                labels
            )

            total_loss += loss.item()

            preds = torch.argmax(
                outputs,
                dim=1
            )

            correct += (
                preds == labels
            ).sum().item()

            total += labels.size(0)

    accuracy = correct / total

    return (
        total_loss / len(data_loader),
        accuracy
    )

In [19]:
# TRAINING THE MODEL
EPOCHS = 3

for epoch in range(EPOCHS):

    train_loss, train_acc = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss, val_acc = eval_epoch(
        model,
        val_loader,
        criterion,
        device
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS}"
    )

    print(
        f"Train Loss: {train_loss:.4f}"
    )

    print(
        f"Train Acc : {train_acc:.4f}"
    )

    print(
        f"Val Loss  : {val_loss:.4f}"
    )

    print(
        f"Val Acc   : {val_acc:.4f}"
    )

    print("-"*40)

Epoch 1/3
Train Loss: 0.6927
Train Acc : 0.5426
Val Loss  : 0.6276
Val Acc   : 0.6525
----------------------------------------
Epoch 2/3
Train Loss: 0.6501
Train Acc : 0.6362
Val Loss  : 0.6420
Val Acc   : 0.6356
----------------------------------------
Epoch 3/3
Train Loss: 0.5746
Train Acc : 0.6915
Val Loss  : 0.5923
Val Acc   : 0.7203
----------------------------------------


In [20]:
import os

os.makedirs(
    "/content/drive/MyDrive/MultilingualFakeNews/models",
    exist_ok=True
)

print("Directory created.")

Directory created.


In [21]:
os.listdir(
    "/content/drive/MyDrive/MultilingualFakeNews"
)

['data', 'models']

In [22]:
# SAVING THE TRAINED MODEL
torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/MultilingualFakeNews/models/xlmr_baseline.pt"
)

In [23]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "model_name": "xlm-roberta-base",
        "max_length": 384
    },
    "/content/drive/MyDrive/MultilingualFakeNews/models/xlmr_baseline_checkpoint.pt"
)

Running Diagnosis as the Epoch values reported shows something is wrong with the training setup


In [24]:
# Step1: Verifying labels
print(df["target"].value_counts())

target
0    298
1    290
Name: count, dtype: int64


In [25]:
# Step2: Inspecting a training batch
batch = next(iter(train_loader))

print(batch["label"])

tensor([1, 1, 1, 1, 0, 1, 1, 0])


In [26]:
# Step3: Inspect raw model outputs
with torch.no_grad():
    outputs = model(
        batch["input_ids"].to(device),
        batch["attention_mask"].to(device)
    )

print(outputs[:5])
print(torch.argmax(outputs, dim=1))

tensor([[ 0.0081,  0.8784],
        [ 0.1255,  0.6056],
        [ 0.2345,  0.6246],
        [ 0.1639,  0.7142],
        [ 0.7155, -0.0128]], device='cuda:0')
tensor([1, 1, 1, 1, 0, 1, 1, 0], device='cuda:0')


In [27]:
# Step4: Verify trainable parameters
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total:", total_params)
print("Trainable:", trainable_params)

Total: 278045186
Trainable: 278045186


In [28]:
EPOCHS = 10

In [29]:
best_val_acc = 0

for epoch in range(EPOCHS):

    train_loss, train_acc = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss, val_acc = eval_epoch(
        model,
        val_loader,
        criterion,
        device
    )

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Acc: {train_acc:.4f}")
    print(f"Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            "/content/drive/MyDrive/MultilingualFakeNews/models/best_xlmr.pt"
        )

        print("Best model saved.")

    print("-"*40)

Epoch 1/10
Train Acc: 0.8213
Val Acc: 0.7542
Best model saved.
----------------------------------------
Epoch 2/10
Train Acc: 0.8872
Val Acc: 0.7288
----------------------------------------
Epoch 3/10
Train Acc: 0.9574
Val Acc: 0.7203
----------------------------------------
Epoch 4/10
Train Acc: 0.9468
Val Acc: 0.7542
----------------------------------------
Epoch 5/10
Train Acc: 0.9894
Val Acc: 0.7881
Best model saved.
----------------------------------------
Epoch 6/10
Train Acc: 0.9851
Val Acc: 0.7542
----------------------------------------
Epoch 7/10
Train Acc: 0.9830
Val Acc: 0.7203
----------------------------------------
Epoch 8/10
Train Acc: 0.9894
Val Acc: 0.8390
Best model saved.
----------------------------------------
Epoch 9/10
Train Acc: 0.9723
Val Acc: 0.7797
----------------------------------------
Epoch 10/10
Train Acc: 0.9936
Val Acc: 0.8051
----------------------------------------


In [30]:
import os

path = "/content/drive/MyDrive/MultilingualFakeNews/models/best_xlmr.pt"

print(os.path.exists(path))

True


In [31]:
import json

metadata = {
    "model": "xlm-roberta-base",
    "max_length": 384,
    "best_epoch": 8,
    "validation_accuracy": 0.8390,
    "dataset": "TALLIP English",
    "classes": {
        "0": "Legit",
        "1": "Fake"
    }
}

with open(
    "/content/drive/MyDrive/MultilingualFakeNews/models/best_xlmr_metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=4)